# B-Free x GlobalForge — Notebook 03: Evaluation (K3)

**Target B: RTX PRO 6000, hoàn toàn OFFLINE.** Đánh giá **3 models** trên **17 in-the-wild subsets** + **standard benchmarks** (AIGCDetect, GenImage, UnivFD, DRCT, Synthbuster, EvalGEN):

| # | Model | Protocol |
|---|-------|----------|
| 1 | Integrated LoRA (output K2) — **bắt buộc** | multi-crop 504: replicate-pad ảnh < 504, 5 crops (center + 4 corners), average logits |
| 2 | B-Free baseline `BFREE_dino2reg4` | Wrapper5crops gốc của repo (replicate + 5-crop ở mức embedding), num_classes=1 |
| 3 | GlobalForge released `REM vit_l_a` | **protocol code thật** `eval_in_the_wild.py`: short-side ≤ 1296 → center-crop 224, ngược lại resize 1296 rồi crop; PNG → JPEG q100; softmax → log-odds |

Score = `logit_fake − logit_real`; model 3 quy về logit `log(p/(1-p))`. Metrics: AUC, bAcc, NLL, ECE, Pd10, EER (`utils/dmetrics.py`). CO-SPY: `fake_only` (paper default). Grouped average theo parent dataset (giống `combine_parent_dataset_average`).

**Inputs (attach trước khi chạy):**
1. **Output notebook 01** (wheels + repo source)
2. **Output notebook 02** (checkpoint LoRA) — qua *Add Input → Your Work*
3. `bfree-baseline-weights` — thư mục `BFREE_dino2reg4/` (config.yaml + weights .pth)
4. `globalforge-code` (thư mục `code/` có `models/REM.py`) + `globalforge-backbone-vitla` (HF ViT-L `vit_l_a/`) + `globalforge-weights` (`checkpoint-best.pth.part_*` × 13, ~1.25GB)
5. `wild-benchmarks` — DATA_ROOT 17 subsets layout `eval_in_the_wild.py` (Chameleon, synthwildx/…, WildRF/test/…, AIGIBench/…, CO-SPY-In-the-Wild/…, RRDataset, B-Free, realchain_CD; mỗi cái `0_real/` + `1_fake/`)

> Models 2/3: nếu thiếu input thì **skip có cảnh báo** (đặt `SKIP_MISSING_MODELS = True`) — kết quả vẫn ra cho các model có sẵn; đặt `False` để fail sớm.

In [ ]:
import logging
import os
import sys
from pathlib import Path

logger = logging.getLogger("bfree")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_console = logging.StreamHandler()
_console.setFormatter(logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S"))
logger.addHandler(_console)

logger.info("stdlib + logging ready")

In [ ]:
import glob
from pathlib import Path

WORK = Path("/kaggle/working")


def find_one(patterns, what, optional=False):
    for pat in patterns:
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0]
    if optional:
        return None
    raise FileNotFoundError(f"NOT FOUND: {what}\nsearched: {list(patterns)}")


def find_dir_containing(marker_rel, patterns, what, optional=False):
    for pat in patterns:
        for hit in sorted(glob.glob(pat)):
            root = Path(hit)
            while root != Path("/kaggle/input") and root != root.parent:
                if (root / marker_rel).is_file():
                    return root
                root = root.parent
    if optional:
        return None
    raise FileNotFoundError(f"NOT FOUND: {what} (marker: {marker_rel})")


SKIP_MISSING_MODELS = True   # False = fail sớm nếu model 2/3 thiếu input

# ---- bundle cua notebook 01 ----
WHEELS_DIR = find_one(
    ["/kaggle/input/*/wheels_rtxpro6000", "/kaggle/input/wheels_rtxpro6000",
     "/kaggle/input/*/wheels*", "/kaggle/input/wheels*"],
    "wheel bundle (output of notebook 01)")
REPO_SRC = find_dir_containing(
    "code/networks/bfree_globalforge_vit.py",
    ["/kaggle/input/*/bfree_src/code/networks/bfree_globalforge_vit.py",
     "/kaggle/input/bfree_src*/code/networks/bfree_globalforge_vit.py",
     "/kaggle/input/*/B-Free/code/networks/bfree_globalforge_vit.py",
     "/kaggle/input/B-Free*/code/networks/bfree_globalforge_vit.py",
     "/kaggle/input/*/code/networks/bfree_globalforge_vit.py"],
    "B-Free repo source (output of notebook 01)")

# ---- K2 checkpoint (output notebook 02) ----
K2_CKPT = find_one(
    ["/kaggle/input/*/bfree_globalforge_lora_r16.pth",
     "/kaggle/input/bfree_globalforge_lora_r16.pth",
     "/kaggle/input/*/bfree_globalforge_lora_r16_best.pth",
     "/kaggle/input/bfree_globalforge_lora_r16_best.pth"],
    "K2 LoRA checkpoint (output of notebook 02)")

# ---- optional: B-Free baseline weights ----
BFREE_WEIGHTS_DIR = find_dir_containing(
    "config.yaml",
    ["/kaggle/input/*/BFREE_dino2reg4/config.yaml",
     "/kaggle/input/BFREE_dino2reg4/config.yaml",
     "/kaggle/input/*/bfree-baseline-weights/BFREE_dino2reg4/config.yaml"],
    "B-Free baseline weights (BFREE_dino2reg4/)", optional=SKIP_MISSING_MODELS)

# ---- optional: GlobalForge code + backbone + ckpt parts ----
GF_CODE_DIR = find_dir_containing(
    "models/REM.py",
    ["/kaggle/input/*/code/models/REM.py",
     "/kaggle/input/code/models/REM.py",
     "/kaggle/input/*/globalforge*/code/models/REM.py"],
    "GlobalForge code (models/REM.py)", optional=SKIP_MISSING_MODELS)
GF_BACKBONE_VITLA = find_one(
    ["/kaggle/input/*/vit_l_a/config.json", "/kaggle/input/vit_l_a/config.json",
     "/kaggle/input/*/globalforge-backbone-vitla/vit_l_a/config.json"],
    "GlobalForge HF backbone vit_l_a/", optional=SKIP_MISSING_MODELS)
GF_CKPT_PARTS_DIR = find_one(
    ["/kaggle/input/*/checkpoint-best.pth.part_aa", "/kaggle/input/checkpoint-best.pth.part_aa",
     "/kaggle/input/*/globalforge-weights/checkpoint-best.pth.part_aa"],
    "GlobalForge checkpoint parts (checkpoint-best.pth.part_*)", optional=SKIP_MISSING_MODELS)
GF_CKPT_ASSEMBLED = find_one(
    ["/kaggle/input/*/checkpoint-best.pth", "/kaggle/input/checkpoint-best.pth"],
    "assembled GlobalForge checkpoint", optional=SKIP_MISSING_MODELS)

# ---- wild + standard benchmarks root ----
WILD_MARKERS = ["Chameleon/0_real", "synthwildx", "WildRF/test", "AIGIBench",
                "CO-SPY-In-the-Wild", "RRDataset", "realchain_CD"]
DATA_ROOT = None
for cand in sorted(glob.glob("/kaggle/input/*/")) + ["/kaggle/input/"]:
    if any(Path(cand, m).exists() for m in WILD_MARKERS):
        DATA_ROOT = Path(cand)
        break
if DATA_ROOT is None:
    raise FileNotFoundError("wild benchmarks root not found under /kaggle/input — attach 'wild-benchmarks'.")

wheels = sorted(glob.glob(str(Path(WHEELS_DIR) / "*.whl")))
assert wheels, f"No .whl files inside {WHEELS_DIR}"
logger.info(f"WHEELS_DIR   = {WHEELS_DIR} ({len(wheels)} wheels)")
logger.info(f"REPO_SRC     = {REPO_SRC}")
logger.info(f"K2_CKPT      = {K2_CKPT}")
logger.info(f"BFREE_WEIGHT = {BFREE_WEIGHTS_DIR}")
logger.info(f"GF_CODE      = {GF_CODE_DIR}")
logger.info(f"GF_BACKBONE  = {GF_BACKBONE_VITLA}")
logger.info(f"GF_PARTS     = {GF_CKPT_PARTS_DIR} | assembled={GF_CKPT_ASSEMBLED}")
logger.info(f"DATA_ROOT    = {DATA_ROOT}")

In [ ]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

logger.info("offline env vars set (HF hub blocked, bytecode off — repo source stays read-only in input)")

In [ ]:
!pip install --no-index --find-links="{WHEELS_DIR}" \
    torch torchvision timm peft transformers accelerate \
    pandas numpy matplotlib seaborn scikit-learn scipy \
    pyyaml pillow tqdm safetensors huggingface_hub

import torch
import timm
import peft
import transformers

assert torch.__version__.startswith("2.8.0"), f"torch {torch.__version__} != 2.8.0+cu128 (bundle wrong?)"
assert transformers.__version__ == "4.55.4", f"transformers {transformers.__version__} != 4.55.4"
assert peft.__version__ == "0.15.2", f"peft {peft.__version__} != 0.15.2"
import pandas
assert pandas.__version__.split(".")[0] == "2", "pandas>=3 breaks sklearn (risk table)"
logger.info(f"pip --no-index OK | torch={torch.__version__} timm={timm.__version__} "
            f"peft={peft.__version__} transformers={transformers.__version__}")

In [ ]:
import sys

sys.path.insert(0, str(Path(REPO_SRC) / "code"))

stubs = sorted(glob.glob(str(Path(REPO_SRC) / "code" / "modules" / "*_stub.py")))
assert not stubs, f"Stub files still present (K0 not merged?): {stubs}"

from networks.bfree_globalforge_vit import BFreeGlobalForgeViT
from configs.loader import load_config, build_model_kwargs
from datasets.bfree_dataset import BFreeDataset, DegradationPipeline
from modules.lib_adapter import LIBAdapter
from modules.gsr_adapter import GSRAdapter
from modules.dcs_loss import DCSLoss, info_nce_loss
import networks.bfree_globalforge_vit as _bgv

logger.info(f"repo import OK from {_bgv.__file__} (no stubs — K0 verified)")

### Model 1 — Integrated LoRA (K2 output, bắt buộc)

Tái tạo đúng kiến trúc từ CONFIG trong checkpoint (kể cả LoRA wrap cùng rank) rồi load state dict.

In [ ]:
import torch


def load_integrated_lora(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ckpt.get("config", {})
    m = BFreeGlobalForgeViT(
        arch=cfg.get("arch", "vit_base_patch14_reg4_dinov2.lvd142m"),
        num_classes=cfg.get("num_classes", 2),
        img_size=cfg.get("img_size", 504),
        pretrained=False, use_lib=True, use_gsr=True, use_dcs=True,
        lib_kernel=cfg.get("lib_kernel", 3), lib_tau=cfg.get("lib_tau", 0.5),
        gsr_window=cfg.get("gsr_window", 3), gsr_mask_prob=cfg.get("gsr_mask_prob", 1.0),
        dcs_tau=cfg.get("dcs_tau", 0.07), lambda_dcs=cfg.get("lambda_dcs", 0.01),
        label_smoothing=cfg.get("label_smoothing", 0.1),
    )
    from train_lora import apply_lora_to_backbone
    m = apply_lora_to_backbone(m, r=cfg.get("lora_rank", 16))
    report = m.load_state_dict(ckpt["model"], strict=False)
    logger.info(f"Integrated LoRA: missing={len(report.missing_keys)} "
                f"unexpected={len(report.unexpected_keys)} epoch={ckpt.get('epoch')} "
                f"val_bAcc={ckpt.get('val_bacc', 'n/a')}")
    assert not report.unexpected_keys, f"unexpected keys: {report.unexpected_keys[:10]}"
    return m.to(device).eval()

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
model_integrated = load_integrated_lora(K2_CKPT, DEVICE)
logger.info("Model 1 (Integrated LoRA) loaded.")

### Model 2 — B-Free baseline (`BFREE_dino2reg4`) — nguyên protocol repo

`get_network` + `load_weights` + `Wrapper5crops` (tự replicate + 5-crop ở mức patch-embedding); feed full ảnh qua ToTensor+Normalize theo `norm_type` trong config.yaml (giống `main_bfree.py`).

In [ ]:
model_bfree = None
BFREE_NORM = "resnet"
if BFREE_WEIGHTS_DIR is None:
    logger.warning("[SKIP] Model 2 (B-Free baseline): weights input not attached")
elif not SKIP_MISSING_MODELS:
    raise FileNotFoundError("B-Free baseline weights required but not attached")
else:
    import yaml

    from networks import get_network, load_weights

    wdir = Path(BFREE_WEIGHTS_DIR)
    with open(wdir / "config.yaml") as f:
        bcfg = yaml.safe_load(f)
    model_bfree = load_weights(get_network(bcfg["arch"]), str(wdir / bcfg["weights_file"]))
    model_bfree = model_bfree.to(DEVICE).eval()
    BFREE_NORM = bcfg["norm_type"]
    logger.info(f"Model 2 (B-Free baseline) loaded: arch={bcfg['arch']} norm={BFREE_NORM}")

### Model 3 — GlobalForge released (`REM vit_l_a`, reassemble 13 parts)

Ghép `checkpoint-best.pth.part_*` → `checkpoint-best.pth` (~1.25GB), suy ra lora_rank/use_lib/use_gsr từ state dict, dựng `REM(vit_l_a)` với `GLOBALFORGE_WEIGHTS_DIR` trỏ backbone HF local, load weights.

In [ ]:
model_gf = None
if GF_CODE_DIR is None or (GF_CKPT_PARTS_DIR is None and GF_CKPT_ASSEMBLED is None):
    logger.warning("[SKIP] Model 3 (GlobalForge REM): code/weights/backbone input not attached")
elif not SKIP_MISSING_MODELS:
    raise FileNotFoundError("GlobalForge inputs required but not attached")
else:
    import sys as _sys

    _sys.path.insert(0, str(GF_CODE_DIR))
    if GF_CKPT_ASSEMBLED is None:
        GF_CKPT = WORK / "checkpoint-best.pth"
        if not GF_CKPT.exists() or GF_CKPT.stat().st_size < 1_000_000_000:
            parts = sorted(glob.glob(str(Path(GF_CKPT_PARTS_DIR) / "checkpoint-best.pth.part_*")))
            assert parts, f"no checkpoint-best.pth.part_* under {GF_CKPT_PARTS_DIR}"
            logger.info(f"reassembling {len(parts)} parts ...")
            with open(GF_CKPT, "wb") as out:
                for p in parts:
                    with open(p, "rb") as f:
                        while True:
                            chunk = f.read(1024 * 1024 * 64)
                            if not chunk:
                                break
                            out.write(chunk)
        GF_CKPT_ASSEMBLED = GF_CKPT
    logger.info(f"checkpoint: {GF_CKPT_ASSEMBLED} "
                f"({Path(GF_CKPT_ASSEMBLED).stat().st_size/1024**3:.2f} GB)")

    GF_WEIGHTS_ROOT = WORK / "gf_weights"
    GF_WEIGHTS_ROOT.mkdir(exist_ok=True)
    vit_link = GF_WEIGHTS_ROOT / "vit_l_a"
    if not vit_link.exists():
        if Path(GF_BACKBONE_VITLA, "config.json").is_file() and Path(GF_BACKBONE_VITLA).name == "vit_l_a":
            src_backbone = Path(GF_BACKBONE_VITLA)
        elif Path(GF_BACKBONE_VITLA, "vit_l_a", "config.json").is_file():
            src_backbone = Path(GF_BACKBONE_VITLA) / "vit_l_a"
        else:
            src_backbone = None
        if src_backbone is not None:
            import shutil as _shutil
            _shutil.copytree(src_backbone, vit_link)
            logger.info(f"backbone vit_l_a staged at {vit_link}")
    os.environ["GLOBALFORGE_WEIGHTS_DIR"] = str(GF_WEIGHTS_ROOT)

    import models.REM as REM

    ckpt = torch.load(str(GF_CKPT_ASSEMBLED), map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt)
    lora_rank = next((int(v.shape[0]) for k, v in state.items()
                      if "lora_A.default.weight" in k), 16)
    use_lib = any(k.startswith("lib.") for k in state)
    use_gsr = any(k.startswith("gsr.") for k in state)
    logger.info(f"GlobalForge released: lora_rank={lora_rank} use_lib={use_lib} use_gsr={use_gsr}")
    model_gf = REM.__dict__["REM"](
        mode="vit_l_a", use_lib=use_lib, use_gsr=use_gsr,
        lib_layer=0, gsr_layer=0, lib_kernel=3, lib_tau=0.5,
        gsr_window=3, gsr_mask_prob=1.0,
    )
    model_gf.load_state_dict(state)
    model_gf = model_gf.to(DEVICE).eval()
    logger.info("Model 3 (GlobalForge released) loaded.")

### Scoring kernels (protocol code thật của từng model)

- **Model 1**: replicate-pad ảnh < 504 (numpy `edge` — tương đương `replicate_wrap`), 5 crops 504 (center + 4 corners), batch forward, average logits → `l1 − l0`.
- **Model 2**: feed full ảnh, Wrapper5crops tự xử lý; num_classes=1 → score = logit.
- **Model 3**: `short256_center` với `eval_resize_short=1296`; PNG round-trip JPEG q100; `Norm(imagenet)` nằm trong model; softmax → log-odds.

In [ ]:
import io

import numpy as np
from PIL import Image, UnidentifiedImageError
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from utils import dmetrics
from utils.normalization import get_list_norm

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}
GF_EVAL_RESIZE_SHORT = 1296


def list_images(root, limit=None):
    out = [str(p) for p in sorted(Path(root).iterdir())
           if p.is_file() and p.suffix.lower() in IMG_EXT]
    return out[:limit] if limit else out


def replicate_pad_504(img):
    w, h = img.size
    if w >= 504 and h >= 504:
        return img
    arr = np.asarray(img)
    pad_h, pad_w = max(0, 504 - h), max(0, 504 - w)
    if pad_h or pad_w:
        arr = np.pad(arr, ((0, pad_h), (0, pad_w), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
    return img


def five_crop_boxes(img):
    w, h = img.size
    s = 504
    if w < s or h < s:
        raise ValueError(f"image smaller than 504 after padding: {w}x{h}")
    cx, cy = (w - s) // 2, (h - s) // 2
    return [(cx, cy, s, s), (0, 0, s, s), (w - s, 0, s, s), (0, h - s, s, s), (w - s, h - s, s, s)]


@torch.no_grad()
def score_image_integrated(model, img_path, device=DEVICE):
    img = replicate_pad_504(Image.open(img_path).convert("RGB"))
    views = torch.stack([T.Compose(get_list_norm("resnet"))(img.crop(b)) for b in five_crop_boxes(img)])
    logits = model(views.to(device))["logits"].float().mean(dim=0)
    return float(logits[1] - logits[0])


@torch.no_grad()
def score_image_bfree(model, img_path, device=DEVICE, norm_type=BFREE_NORM):
    x = T.Compose(get_list_norm(norm_type))(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    out = model(x)
    if out.shape[1] == 1:
        return float(out[0, 0])
    return float(out[0, 1] - out[0, 0])


@torch.no_grad()
def score_image_globalforge(model, img_path, device=DEVICE):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    short = min(w, h)
    if short > GF_EVAL_RESIZE_SHORT:
        scale = GF_EVAL_RESIZE_SHORT / short
        img = TF.resize(img, [round(h * scale), round(w * scale)],
                        interpolation=T.InterpolationMode.BICUBIC)
    img = TF.center_crop(img, [224, 224])
    if img_path.lower().endswith(".png"):
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=100, optimize=True)
        buf.seek(0)
        img = Image.open(buf).convert("RGB").copy()
    x = T.functional.to_tensor(img).unsqueeze(0).to(device)
    out = model(x)
    logits = (out["logits"] if isinstance(out, dict) else out).float().clamp(-30, 30)
    p = float(torch.softmax(logits, dim=1)[0, 1])
    return float(np.log(p / max(1.0 - p, 1e-12)))


SCORERS = {"Integrated-LoRA": (lambda p: score_image_integrated(model_integrated, p), True)}
if model_bfree is not None:
    SCORERS["B-Free-baseline"] = (lambda p: score_image_bfree(model_bfree, p), True)
if model_gf is not None:
    SCORERS["GlobalForge-REM"] = (lambda p: score_image_globalforge(model_gf, p), True)
logger.info(f"scorers ready: {list(SCORERS)}")

In [ ]:
# ============ Wild subset registry (17 subsets, layout 0_real/1_fake) ============

WILD_SUBSETS = {
    "Chameleon":           ("Chameleon/0_real", "Chameleon/1_fake", False),
    "SynthWildx-DALLE3":   ("synthwildx/dalle3/0_real", "synthwildx/dalle3/1_fake", False),
    "SynthWildx-Firefly":  ("synthwildx/firefly/0_real", "synthwildx/firefly/1_fake", False),
    "SynthWildx-Midj.":    ("synthwildx/midjourney_v5/0_real", "synthwildx/midjourney_v5/1_fake", False),
    "WildRF-FB":           ("WildRF/test/facebook/0_real", "WildRF/test/facebook/1_fake", False),
    "WildRF-Reddit":       ("WildRF/test/reddit/0_real", "WildRF/test/reddit/1_fake", False),
    "WildRF-Twitter":      ("WildRF/test/twitter/0_real", "WildRF/test/twitter/1_fake", False),
    "AIGIBench-SocRF":     ("AIGIBench/SocialRF/0_real", "AIGIBench/SocialRF/1_fake", False),
    "AIGIBench-ComAI":     ("AIGIBench/CommunityAI/0_real", "AIGIBench/CommunityAI/1_fake", False),
    "CO-SPY-Civitai":      ("CO-SPY-In-the-Wild/civitai/0_real", "CO-SPY-In-the-Wild/civitai/1_fake", True),
    "CO-SPY-DALLE3":       ("CO-SPY-In-the-Wild/dalle3/0_real", "CO-SPY-In-the-Wild/dalle3/1_fake", True),
    "CO-SPY-instavibe.ai": ("CO-SPY-In-the-Wild/instavibeai/0_real", "CO-SPY-In-the-Wild/instavibeai/1_fake", True),
    "CO-SPY-Lexica":       ("CO-SPY-In-the-Wild/lexica/0_real", "CO-SPY-In-the-Wild/lexica/1_fake", True),
    "CO-SPY-Midj.v6":      ("CO-SPY-In-the-Wild/midjourney/0_real", "CO-SPY-In-the-Wild/midjourney/1_fake", True),
    "RR-Dataset":          ("RRDataset/0_real", "RRDataset/1_fake", False),
    "BFree-Online":        ("B-Free/0_real", "B-Free/1_fake", False),
    "real-chain":          ("realchain_CD/0_real", "realchain_CD/1_fake", False),
}

STANDARD_BENCHMARKS = {
    "AIGCDetect":  ("AIGCDetect/0_real", "AIGCDetect/1_fake"),
    "GenImage":    ("GenImage/0_real", "GenImage/1_fake"),
    "UnivFD":      ("UnivFD/0_real", "UnivFD/1_fake"),
    "DRCT":        ("DRCT/0_real", "DRCT/1_fake"),
    "Synthbuster": ("Synthbuster/0_real", "Synthbuster/1_fake"),
    "EvalGEN":     ("EvalGEN/0_real", "EvalGEN/1_fake"),
}

available = [k for k, (r, f, _) in WILD_SUBSETS.items()
             if (DATA_ROOT / r).is_dir() and (DATA_ROOT / f).is_dir()]
missing = [k for k in WILD_SUBSETS if k not in available]
logger.info(f"wild subsets available: {len(available)}/17" + (f" (skipped: {missing})" if missing else ""))
assert available, "no wild subset folders found under DATA_ROOT"
MAX_IMAGES = None

In [ ]:
import tqdm


def run_subset(model_name, scorer, real_dir, fake_dir, fake_only=False, max_images=None):
    real_paths = list_images(DATA_ROOT / real_dir, limit=max_images)
    fake_paths = list_images(DATA_ROOT / fake_dir, limit=max_images)
    if fake_only:
        paths, labels = fake_paths, [1] * len(fake_paths)
    else:
        paths = real_paths + fake_paths
        labels = [0] * len(real_paths) + [1] * len(fake_paths)
    if not paths:
        return None

    scores, y, n_skipped = [], [], 0
    for path, label in tqdm.tqdm(list(zip(paths, labels)),
                                 desc=f"{model_name}::{Path(fake_dir).parent.name}",
                                 leave=False):
        try:
            scores.append(scorer(path))
            y.append(label)
        except (OSError, UnidentifiedImageError, ValueError):
            n_skipped += 1
    if n_skipped:
        logger.warning(f"  {n_skipped} unreadable images skipped")
    if not scores:
        return None
    scores = np.asarray(scores, dtype=float)
    y = np.asarray(y, dtype=int)

    if fake_only:
        return {"Model": model_name, "n_images": len(scores), "n_skipped": n_skipped,
                "AUC": float("nan"), "bAcc": float((scores > 0).mean() * 100.0),
                "NLL": float("nan"), "ECE": float("nan"),
                "Pd10": float("nan"), "EER": float("nan"), "fake_only": True}
    return {"Model": model_name, "n_images": len(scores), "n_skipped": n_skipped,
            "AUC": float(dmetrics.roc_auc_score(y, scores)),
            "bAcc": float(dmetrics.balanced_accuracy_score(y, scores > 0) * 100.0),
            "NLL": float(dmetrics.balanced_nll_binary(y, scores)),
            "ECE": float(dmetrics.balanced_ece_binary(y, scores)),
            "Pd10": float(dmetrics.pd_at_far(y, scores, 0.10) * 100.0),
            "EER": float(dmetrics.calculate_eer2(y, scores) * 100.0),
            "fake_only": False}


all_results = []
for subset, (real_dir, fake_dir, fake_only) in WILD_SUBSETS.items():
    if not ((DATA_ROOT / real_dir).is_dir() and (DATA_ROOT / fake_dir).is_dir()):
        logger.info(f"[skip] {subset}: folders missing")
        continue
    for model_name, (scorer, _) in SCORERS.items():
        r = run_subset(model_name, scorer, real_dir, fake_dir,
                       fake_only=fake_only, max_images=MAX_IMAGES)
        if r:
            all_results.append({"Benchmark": subset, **r})
            logger.info(f"{subset:20s} | {model_name:16s} | bAcc={r['bAcc']:6.2f} "
                        f"AUC={r['AUC']:.4f} n={r['n_images']}")

import pandas as pd

wild_df = pd.DataFrame(all_results)
display(wild_df)

In [ ]:
# ============ Standard benchmarks (optional - skip neu chua attach) ============
std_results = []
for bench, (real_dir, fake_dir) in STANDARD_BENCHMARKS.items():
    if not ((DATA_ROOT / real_dir).is_dir() and (DATA_ROOT / fake_dir).is_dir()):
        logger.info(f"[skip] {bench}: folders missing under {DATA_ROOT}")
        continue
    for model_name, (scorer, _) in SCORERS.items():
        r = run_subset(model_name, scorer, real_dir, fake_dir,
                       fake_only=False, max_images=MAX_IMAGES)
        if r:
            std_results.append({"Benchmark": bench, **r})
            logger.info(f"{bench:14s} | {model_name:16s} | bAcc={r['bAcc']:6.2f} "
                        f"AUC={r['AUC']:.4f} n={r['n_images']}")

std_df = pd.DataFrame(std_results)
if len(std_df):
    display(std_df)

In [ ]:
# ============ Combine + grouped parent average + save ============
def get_parent_dataset_key(result_key):
    if result_key.startswith("SynthWildx-"): return "SynthWildx"
    if result_key.startswith("WildRF-"): return "WildRF"
    if result_key.startswith("AIGIBench-"): return "AIGIBench"
    if result_key.startswith("CO-SPY-"): return "CO-SPY"
    if result_key.startswith("BFree-"): return "BFree"
    return result_key


def combine_parent_dataset_average(df, metric="bAcc"):
    '''Trung binh tung parent group, roi trung binh cac group (giong eval_in_the_wild.py).'''
    out = {}
    for model in df["Model"].unique():
        groups = {}
        for subset, value in df[df["Model"] == model].groupby("Benchmark")[metric].last().items():
            groups.setdefault(get_parent_dataset_key(subset), []).append(float(value))
        out[model] = sum(sum(v) / len(v) for v in groups.values()) / len(groups)
    return out


results_df = pd.concat([wild_df, std_df], ignore_index=True)
results_df.to_csv(WORK / "eval_results.csv", index=False)

wild_only = results_df[results_df["Benchmark"].isin(WILD_SUBSETS)]
avg_rows = [{"Model": m, "Benchmark": "Avg B.Acc (wild, parent-avg)", "bAcc": v, "fake_only": False}
            for m, v in combine_parent_dataset_average(wild_only).items()]
results_df = pd.concat([results_df, pd.DataFrame(avg_rows)], ignore_index=True)
results_df.to_csv(WORK / "eval_results.csv", index=False)
logger.info("saved /kaggle/working/eval_results.csv")
display(results_df.pivot_table(index="Benchmark", columns="Model", values="bAcc", dropna=False))

In [ ]:
# ============ Visualization: bar chart + heatmap ============
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = results_df[results_df["Benchmark"] != "Avg B.Acc (wild, parent-avg)"].copy()
piv_bacc = plot_df.pivot_table(index="Benchmark", columns="Model", values="bAcc")

fig, ax = plt.subplots(figsize=(13, max(6, 0.35 * len(piv_bacc))), dpi=130)
piv_bacc.plot(kind="barh", ax=ax)
ax.set_xlabel("bAcc (%)")
ax.set_title("Balanced Accuracy per subset x model")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(WORK / "eval_bacc_bar.png")

fig, ax = plt.subplots(figsize=(9, max(6, 0.35 * len(piv_bacc))), dpi=130)
sns.heatmap(piv_bacc, annot=True, fmt=".1f", cmap="RdYlGn", vmin=50, vmax=100, ax=ax)
ax.set_title("bAcc heatmap (model x subset)")
fig.tight_layout()
fig.savefig(WORK / "eval_bacc_heatmap.png")
logger.info("saved eval_bacc_bar.png + eval_bacc_heatmap.png")

## Evaluation Complete (K3)

Outputs trong `/kaggle/working/`:
- `eval_results.csv` — model × subset × {AUC, bAcc, NLL, ECE, Pd10, EER} + hàng `Avg B.Acc (wild, parent-avg)`
- `eval_bacc_bar.png` / `eval_bacc_heatmap.png`

**K3 checklist:** notebook chạy được · eval results CSV + heatmap + bar chart.

**K4 (post-training analysis):** dùng `train_log.csv` từ K2 — Pearson/Spearman CE vs DCS + loss dynamics (tương đương `eval/analyze_loss.py` của repo), cập nhật vào báo cáo Phase 8.